# CalCOFI DIC 予測(MLP)

方針
- 入力は元の列 + 化学的に意味のある派生特徴 + 観測地点の one-hot。
- 交差検証で残差が 4σ を超えた 17 行は、学習からだけ除く。
- 種(乱数シード)を変えた 20 本の予測を平均して提出する。

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/sample_submission.csv
/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv
/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv


In [2]:
train=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv")
test=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv")
sub=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/sample_submission.csv")

In [3]:
train

,id,Lat_Dec,Lon_Dec,NO2uM,NO3uM,NH3uM,R_TEMP,R_Depth,R_Sal,R_DYNHT,R_Nuts,R_Oxy_micromol.Kg,Unnamed: 12,PO4uM,SiO3uM,TA1.x,Salinity1,Temperature_degC,DIC
0,1,34.385030,-120.665530,0.030,33.80,0.00,7.79,323,141.2,0.642,0.00,37.40948,NaN,2.77,53.86,2287.45,34.1980,7.82,2270.17
1,2,31.418333,-121.998333,0.000,34.70,0.00,7.12,323,140.8,0.767,0.00,64.81441,NaN,2.57,52.50,2279.10,34.0740,7.15,2254.10
2,3,34.385030,-120.665530,0.180,14.20,0.00,11.68,50,246.8,0.144,0.00,180.29150,NaN,1.29,13.01,2230.80,33.5370,11.68,2111.04
3,4,33.482580,-122.533070,0.013,29.67,0.01,8.33,232,158.5,0.562,0.01,89.62595,NaN,2.27,38.98,2265.85,34.0480,8.36,2223.41
4,5,31.414320,-121.997670,0.000,33.10,0.05,7.53,323,143.4,0.740,0.05,60.03062,NaN,2.53,49.28,2278.49,34.1170,7.57,2252.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1449,1450,33.420000,-117.901667,0.000,38.60,0.00,6.52,520,118.2,0.889,0.00,12.17673,NaN,3.11,73.10,2313.52,34.3024,6.57,2311.19
1450,1451,34.271667,-120.023333,0.330,5.90,0.00,13.21,49,290.1,0.149,0.00,224.81120,NaN,0.75,6.50,2223.31,33.3304,13.22,2048.93
1451,1452,31.410000,-121.991667,0.000,33.60,0.00,7.61,322,143.1,0.731,0.00,60.46569,NaN,2.56,49.50,2280.43,34.1269,7.64,2251.34
1452,1453,34.274270,-120.030570,0.517,1.58,0.35,14.47,30,319.0,0.110,0.35,249.28420,NaN,0.54,4.12,2223.88,33.2660,14.47,2030.03


In [4]:
test

,id,Lat_Dec,Lon_Dec,NO2uM,NO3uM,NH3uM,R_TEMP,R_Depth,R_Sal,R_DYNHT,R_Nuts,R_Oxy_micromol.Kg,PO4uM,SiO3uM,TA1,Salinity1,Temperature_degC
0,1455,34.321666,-120.811666,0.020,24.00,0.41,9.51,101,189.9,0.258,0.41,138.838300,1.85,25.50,2244.94,33.830,9.52
1,1456,34.275000,-120.033333,0.000,25.10,0.00,9.84,102,185.2,0.264,0.00,102.709200,2.06,28.30,2253.27,33.963,9.85
2,1457,34.275000,-120.033333,0.000,31.90,0.00,6.60,514,124.1,0.874,0.00,2.174548,3.40,88.10,2316.95,34.241,6.65
3,1458,33.828333,-118.625000,0.000,0.00,0.20,19.21,1,408.1,0.004,0.20,258.674300,0.27,2.50,2240.49,33.465,19.21
4,1459,33.828333,-118.625000,0.020,19.70,0.00,10.65,100,215.5,0.274,0.00,145.839900,1.64,19.40,2238.30,33.720,10.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,1935,31.418030,-121.989970,0.000,0.00,0.01,21.20,2,465.3,0.009,0.01,229.490000,0.28,1.89,2235.34,33.380,21.20
481,1936,31.418030,-121.989970,0.000,24.50,0.00,8.54,232,171.3,0.689,0.00,153.330000,1.81,29.40,2250.00,33.934,8.57
482,1937,31.418030,-121.989970,0.000,33.70,0.00,7.15,323,144.0,0.829,0.00,78.890000,2.51,50.75,2270.19,34.049,7.18
483,1938,32.846330,-117.531300,0.037,0.75,0.05,13.95,30,307.5,0.117,0.05,259.410000,0.49,3.52,2225.36,33.285,13.95


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1454 entries, 0 to 1453
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1454 non-null   int64  
 1   Lat_Dec            1454 non-null   float64
 2   Lon_Dec            1454 non-null   float64
 3   NO2uM              1454 non-null   float64
 4   NO3uM              1454 non-null   float64
 5   NH3uM              1454 non-null   float64
 6   R_TEMP             1454 non-null   float64
 7   R_Depth            1454 non-null   int64  
 8   R_Sal              1454 non-null   float64
 9   R_DYNHT            1454 non-null   float64
 10  R_Nuts             1454 non-null   float64
 11  R_Oxy_micromol.Kg  1454 non-null   float64
 12  Unnamed: 12        0 non-null      float64
 13  PO4uM              1454 non-null   float64
 14  SiO3uM             1454 non-null   float64
 15  TA1.x              1454 non-null   float64
 16  Salinity1          1454 

In [6]:
sub

,id,DIC
0,1455,2150.46882
1,1456,2150.46882
2,1457,2150.46882
3,1458,2150.46882
4,1459,2150.46882
...,...,...
480,1935,2150.46882
481,1936,2150.46882
482,1937,2150.46882
483,1938,2150.46882


In [7]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load data
# train = pd.read_csv("/kaggle/input/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv")
# test = pd.read_csv("/kaggle/input/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- 派生特徴 ----
def add_features(df):
    df = df.copy()
    # 深さは浅いところほど変化が急なので対数も入れる
    df["log_R_Depth"] = np.log1p(df["R_Depth"])
    # 飽和酸素濃度(水温と塩分から求める式)と、見かけの酸素消費量 AOU
    t = df["Temperature_degC"]
    S = df["Salinity1"]
    Ts = np.log((298.15 - t) / (273.15 + t))
    lnO2 = (5.80871 + 3.20291 * Ts + 4.17887 * Ts**2 + 5.10006 * Ts**3
            - 0.0986643 * Ts**4 + 3.80369 * Ts**5
            + S * (-0.00701577 - 0.00770028 * Ts - 0.0113864 * Ts**2 - 0.00951519 * Ts**3)
            - 2.75915e-7 * S**2)
    df["O2sat"] = np.exp(lnO2)
    df["AOU"] = df["O2sat"] - df["R_Oxy_micromol.Kg"]
    # 酸素と栄養塩を組み合わせた保存量(レッドフィールド比 O:P≈170, O:N≈9.3, N:P≈16)
    O = df["R_Oxy_micromol.Kg"]
    df["PO"] = O + 170 * df["PO4uM"]
    df["NO"] = O + 9.3 * df["NO3uM"]
    df["Nstar"] = df["NO3uM"] - 16 * df["PO4uM"]
    df["PO4pre"] = df["PO4uM"] - df["AOU"] / 170
    return df

# ---- 学習から除く外れ行 ----
# 派生特徴なしの基本モデルで交差検証をしたとき、検証の残差が 4σ(残差の標準偏差 約5.25 の4倍 ≈ 21)を
# 超えた行。
OUTLIER_IDS = [34, 91, 250, 269, 419, 462, 510, 736, 748, 773, 918, 972, 1080, 1087, 1169, 1350, 1410]
is_outlier = train["id"].isin(OUTLIER_IDS).values
clean_idx = np.where(~is_outlier)[0]
print("学習に使う行:", len(clean_idx), "/", len(train))

# ---- 観測地点(緯度・経度の組)の one-hot。train に出てくる地点だけで作る ----
site_train = (train["Lat_Dec"].round(4).astype(str) + "_" + train["Lon_Dec"].round(4).astype(str)).values
site_test = (test["Lat_Dec"].round(4).astype(str) + "_" + test["Lon_Dec"].round(4).astype(str)).values
sites = sorted(set(site_train))
site_id = {s: i for i, s in enumerate(sites)}
n_sites = len(sites)
site_onehot_train = np.zeros((len(train), n_sites), dtype=np.float32)
site_onehot_train[np.arange(len(train)), [site_id[s] for s in site_train]] = 1
site_onehot_test = np.zeros((len(test), n_sites), dtype=np.float32)
for r, s in enumerate(site_test):
    if s in site_id:  # train に無い地点は全 0 のまま
        site_onehot_test[r, site_id[s]] = 1
print("地点数:", n_sites, " testのうちtrainに無い地点の行:", int((site_onehot_test.sum(1) == 0).sum()))

train = add_features(train)
test = add_features(test)
test_ids = test["id"].values

# Drop unneeded columns and handle missing values
train = train.drop(columns=["Unnamed: 12", "id"])  # Dropping unnecessary columns

# Find common columns between train and test (excluding "DIC" from train)
common_columns = train.drop(columns=["DIC"]).columns.intersection(test.columns)

# Select the common columns for both train and test
X = train[common_columns]
y = train["DIC"]
test = test[common_columns]
print("入力の列:", list(common_columns))

# Replace NaNs with the column mean
X = X.fillna(X.mean())
test = test.fillna(test.mean())

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

# 目的変数も標準化して学習し、予測を元の単位に戻す
y_mean, y_std = y.mean(), y.std(ddof=0)
y_scaled = ((y - y_mean) / y_std).values

device: cuda
学習に使う行: 1437 / 1454
地点数: 258  testのうちtrainに無い地点の行: 4
入力の列: ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'NH3uM', 'R_TEMP', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Nuts', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'Salinity1', 'Temperature_degC', 'log_R_Depth', 'O2sat', 'AOU', 'PO', 'NO', 'Nstar', 'PO4pre']


In [8]:
# 学習データは小さい(約1400行)ので、DataLoader を使わず GPU 上のテンソルから直接ミニバッチを切り出す
X_all = torch.tensor(np.hstack([X_scaled, site_onehot_train]), dtype=torch.float32, device=device)
y_all = torch.tensor(y_scaled, dtype=torch.float32, device=device)
X_test_all = torch.tensor(np.hstack([test_scaled, site_onehot_test]), dtype=torch.float32, device=device)
n_base = X_scaled.shape[1]  # 地点以外の列の数(この後ろが地点の one-hot)
print(X_all.shape, X_test_all.shape)

torch.Size([1454, 280]) torch.Size([485, 280])


In [9]:
class MLPModel(nn.Module):
    def __init__(self, input_size):
        super(MLPModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 1)  # Output layer for regression

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

In [10]:
import torch.optim as optim

EPOCHS = 3000
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 3e-4
SITE_DROPOUT = 0.5

# Loss function and optimizer
criterion = nn.MSELoss()

# Training function
def train_model(X_in, train_idx, seed, epochs=EPOCHS):
    """train_idx の行で1本のMLPを学習して返す。X_in の後ろ n_sites 列が地点の one-hot。"""
    torch.manual_seed(seed)
    model = MLPModel(input_size=X_in.shape[1]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # 学習率を余弦カーブで0へ下げる
    idx = torch.tensor(train_idx, device=device)
    for epoch in range(epochs):
        model.train()
        perm = idx[torch.randperm(len(idx), device=device)]
        for s in range(0, len(perm), BATCH_SIZE):
            b = perm[s:s + BATCH_SIZE]
            X_batch, y_batch = X_in[b], y_all[b]
            # 地点ドロップアウト: 行ごとに確率 SITE_DROPOUT で地点の列をまとめて 0 にする
            keep = (torch.rand(len(b), 1, device=device) > SITE_DROPOUT).float()
            X_batch = torch.cat([X_batch[:, :n_base], X_batch[:, n_base:] * keep], dim=1)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(1), y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()
    model.eval()
    return model

def predict(model, X_in):
    with torch.no_grad():
        return model(X_in).squeeze(1).cpu().numpy() * y_std + y_mean

## 検証(4分割 × 2回)
- 地点の one-hot は、その分割の学習側に出てくる地点だけ 1 にする(検証側にしかない地点は全 0 = test と同じ状況)。
- 外れ 17 行は学習側からだけ除き、検証の誤差には含める。
- RUN_CV = False にすると飛ばして本番の学習だけ行う。

In [11]:
RUN_CV = True

if RUN_CV:
    import time
    t0 = time.time()
    site_index_train = np.array([site_id[s] for s in site_train])
    oof = np.full((2, len(y)), np.nan)
    fold = 0
    for rep in range(2):
        for tr, va in KFold(4, shuffle=True, random_state=100 + rep).split(X_scaled):
            seen = np.zeros(n_sites, dtype=np.float32)
            seen[site_index_train[tr]] = 1
            X_fold = X_all.clone()
            X_fold[:, n_base:] *= torch.tensor(seen, device=device)  # 学習側に無い地点の列は 0
            tr_clean = np.setdiff1d(tr, np.where(is_outlier)[0])
            model = train_model(X_fold, tr_clean, seed=fold)
            oof[rep, va] = predict(model, X_fold)[va]
            fold += 1
            print(f"fold {fold}/8 done ({time.time() - t0:.0f}s)", flush=True)
    yv = y.values
    rmse_all = np.sqrt(np.mean([(o - yv) ** 2 for o in oof]))
    rmse_clean = np.sqrt(np.mean([(o[~is_outlier] - yv[~is_outlier]) ** 2 for o in oof]))
    print(f"CV RMSE 全行: {rmse_all:.3f}   外れ17行を除いた行: {rmse_clean:.3f}")

fold 1/8 done (86s)
fold 2/8 done (164s)
fold 3/8 done (243s)
fold 4/8 done (321s)
fold 5/8 done (400s)
fold 6/8 done (479s)
fold 7/8 done (557s)
fold 8/8 done (636s)
CV RMSE 全行: 4.833   外れ17行を除いた行: 2.787


## 本番: 外れ以外の train 全行で 20 本学習して平均

In [12]:
import time
N_SEEDS = 20
t0 = time.time()
test_preds = []
for seed in range(N_SEEDS):
    model = train_model(X_all, clean_idx, seed=seed)
    test_preds.append(predict(model, X_test_all))
    print(f"seed {seed + 1}/{N_SEEDS} done ({time.time() - t0:.0f}s)", flush=True)
test_preds = np.array(test_preds)
print("種どうしの予測のばらつき(標準偏差の平均):", round(float(test_preds.std(0).mean()), 3))

seed 1/20 done (106s)
seed 2/20 done (212s)
seed 3/20 done (317s)
seed 4/20 done (422s)
seed 5/20 done (528s)
seed 6/20 done (632s)
seed 7/20 done (738s)
seed 8/20 done (845s)
seed 9/20 done (950s)
seed 10/20 done (1056s)
seed 11/20 done (1161s)
seed 12/20 done (1266s)
seed 13/20 done (1370s)
seed 14/20 done (1479s)
seed 15/20 done (1587s)
seed 16/20 done (1694s)
seed 17/20 done (1801s)
seed 18/20 done (1907s)
seed 19/20 done (2013s)
seed 20/20 done (2118s)
種どうしの予測のばらつき(標準偏差の平均): 0.417


In [13]:
# Make predictions
predictions = test_preds.mean(axis=0)

# Prepare submission
submission = pd.DataFrame({"id": test_ids, "DIC": predictions})
submission.to_csv("submission.csv", index=False)
print(submission.shape, "NaN:", int(submission["DIC"].isna().sum()))
submission.head()

(485, 2) NaN: 0


,id,DIC
0,1455,2171.797685
1,1456,2194.484788
2,1457,2327.010415
3,1458,1992.645246
4,1459,2150.011351


In [14]:
# score: 4.10743
# name:
# student_id: